# 🔍 RAG Pipeline Baseline
Pipeline: **Query Rewrite → Hybrid Search (BM25 + Semantic + RRF) → Rerank → Output**

## 1. Setup & Config

In [1]:
import sys
import os
import time
import json
from pathlib import Path

# Project root
PROJECT_DIR = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_DIR))

print(f"📁 Project dir: {PROJECT_DIR}")

# Data paths
CHUNKS_PATH = str(PROJECT_DIR / "data" / "rag_chunks_v2.json")
EMBEDDINGS_PATH = str(PROJECT_DIR / "data" / "embeddings.npy")

print(f"📄 Chunks: {CHUNKS_PATH}")
print(f"📄 Embeddings: {EMBEDDINGS_PATH}")

📁 Project dir: c:\Users\Admin\OneDrive - Hanoi University of Science and Technology\Desktop\ĐATN
📄 Chunks: c:\Users\Admin\OneDrive - Hanoi University of Science and Technology\Desktop\ĐATN\data\rag_chunks_v2.json
📄 Embeddings: c:\Users\Admin\OneDrive - Hanoi University of Science and Technology\Desktop\ĐATN\data\embeddings.npy


In [2]:
from dotenv import load_dotenv
load_dotenv(PROJECT_DIR / ".env")
print(f"🔑 API Key loaded: {'✅' if os.getenv('GENAI_API_KEY') else '❌'}")

🔑 API Key loaded: ✅


## 2. Initialize Components

### 2.1 CustomSearch (BM25 + Semantic + RRF)

In [3]:
from retrieve_rebuild import CustomSearch

t0 = time.time()
searcher = CustomSearch(
    chunks_path=CHUNKS_PATH,
    embeddings_path=EMBEDDINGS_PATH,
)
print(f"⏱️ Search init: {time.time()-t0:.2f}s")

CustomSearch initialized: 2348 docs, vocab=17139, avgdl=140.9
⏱️ Search init: 0.16s


### 2.2 Reranker

In [8]:
from src.rag.reranker import Reranker

reranker = Reranker(model_name="AITeamVN/Vietnamese_Reranker", device="cuda")
print("✅ Reranker ready (lazy load)")

✅ Reranker ready (lazy load)


### 2.3 Query Rewriter

In [9]:
from src.rag.query_rewriter import QueryRewriter

try:
    rewriter = QueryRewriter()
    USE_REWRITER = True
    print("✅ QueryRewriter ready")
except ValueError as e:
    USE_REWRITER = False
    print(f"⚠️ QueryRewriter disabled: {e}")

✅ QueryRewriter ready


## 3. Pipeline Function

In [10]:
def rag_pipeline(query: str, top_n_search: int = 30, top_k_final: int = 5,
                 use_rewrite: bool = True, verbose: bool = True):
    """
    Full RAG pipeline:
        1. Query Rewrite (optional)
        2. Hybrid Search (BM25 + Semantic + RRF) per query
        3. Rerank all results
        4. Filter & return top_k
    """
    timings = {}

    # ========== STEP 1: Query Rewrite ==========
    t0 = time.time()
    if use_rewrite and USE_REWRITER:
        queries = rewriter.rewrite(query, memory_state=[])
        queries.insert(0, query)
        queries = list(dict.fromkeys(queries))  # Deduplicate
    else:
        queries = [query]
    timings["rewrite"] = time.time() - t0

    if verbose:
        print(f"\n{'='*60}")
        print(f"🔍 Query gốc: '{query}'")
        if len(queries) > 1:
            print(f"📝 Rewritten queries ({len(queries)}):")
            for i, q in enumerate(queries):
                print(f"   [{i}] {q}")

    # ========== STEP 2: Search per query ==========
    t0 = time.time()
    all_results = {}
    for q in queries:
        results = searcher.search(q, top_k=top_n_search // len(queries), top_n=top_n_search)
        for r in results:
            doc_id = r["doc_id"]
            if doc_id not in all_results or r["score"] > all_results[doc_id]["score"]:
                all_results[doc_id] = r
    search_results = sorted(all_results.values(), key=lambda x: x["score"], reverse=True)
    timings["search"] = time.time() - t0

    if verbose:
        print(f"\n📊 Search: {len(search_results)} unique docs (⏱️ {timings['search']:.3f}s)")

    # ========== STEP 3: Rerank ==========
    t0 = time.time()
    reranked = reranker.rerank(query, search_results, top_n=top_k_final * 2)
    timings["rerank"] = time.time() - t0

    if verbose:
        print(f"🔄 Rerank: {len(reranked)} docs (⏱️ {timings['rerank']:.3f}s)")

    # ========== STEP 4: Filter ==========
    final = reranker.filter_context(reranked, min_len=30, top_n=top_k_final)
    total_time = sum(timings.values())

    if verbose:
        print(f"\n✅ Final: {len(final)} docs (Total: ⏱️ {total_time:.3f}s)")
        print(f"   rewrite={timings['rewrite']:.3f}s | search={timings['search']:.3f}s | rerank={timings['rerank']:.3f}s")
        print(f"\n{'─'*60}")
        for i, r in enumerate(final):
            print(f"\n  === Result [{i+1}] ===")
            print(json.dumps(r, ensure_ascii=False, indent=4))


    return final

## 4. Test — Câu hỏi kiến thức cơ bản

In [12]:
results1 = rag_pipeline("mạng máy tính là gì")


🔍 Query gốc: 'mạng máy tính là gì'
📝 Rewritten queries (4):
   [0] mạng máy tính là gì
   [1] khái niệm mạng máy tính, định nghĩa mạng máy tính, kết nối thiết bị
   [2] phân loại mạng máy tính, các loại mạng máy tính, mạng cục bộ LAN, mạng diện rộng WAN
   [3] ứng dụng mạng máy tính, ví dụ về mạng máy tính, internet là gì

📊 Search: 17 unique docs (⏱️ 0.301s)
🔄 Rerank: 10 docs (⏱️ 10.882s)

✅ Final: 5 docs (Total: ⏱️ 11.931s)
   rewrite=0.748s | search=0.301s | rerank=10.882s

────────────────────────────────────────────────────────────

  === Result [1] ===
{
    "doc_id": 1619,
    "score": 0.032018442622950824,
    "content": "Mạng máy tính là một hệ thống các thiết bị số được kết nối với nhau để truyền dữ liệu và trao đổi thông tin. Các thiết bị số trong mạng có thể kết nối với nhau bằng dây cáp mạng (mạng có dây) hoặc bằng sóng vô tuyến (mạng không dây).\n**Cáp mạng** là một loại dây dẫn có vỏ bọc bảo vệ bên ngoài và bên trong có dây dẫn kim loại để truyền tín hiệu điện. Một loại

## 5. Test — Câu hỏi so sánh

In [10]:
results2 = rag_pipeline("so sánh mạng LAN và WAN")


🔍 Query gốc: 'so sánh mạng LAN và WAN'
📝 Rewritten queries (4):
   [0] so sánh mạng LAN và WAN
   [1] so sánh mạng LAN mạng WAN đặc điểm khác biệt
   [2] phân loại mạng máy tính mạng LAN mạng WAN
   [3] cấu trúc mạng LAN mạng WAN phạm vi địa lý

📊 Search: 13 unique docs (⏱️ 0.183s)
🔄 Rerank: 10 docs (⏱️ 17.838s)

✅ Final: 5 docs (Total: ⏱️ 18.732s)
   rewrite=0.711s | search=0.183s | rerank=17.838s

────────────────────────────────────────────────────────────

  === Result [1] ===
{
    "doc_id": 488,
    "score": 0.03125,
    "content": "1.  Phạm vi sử dụng của Internet là:\n    A. Chỉ trong gia đình. B. Chỉ trong một cơ quan. C. Toàn cầu.\n2.  Điện thoại thông minh được kết nối với Internet bằng cách nào?\n    A. Qua dịch vụ 3G, 4G, 5G. B. Kết nối gián tiếp qua wifi. C. Cả A và B.\n\nTheo phạm vi địa lí, các mạng máy tính có thể chia thành hai loại là mạng cục bộ (Local Area Network, viết tắt là **LAN**) và mạng diện rộng (Wide Area Network, viết tắt là **WAN**).\n\n**Mạng LAN** có 

## 6. Test — Câu hỏi cụ thể

In [ ]:
results3 = rag_pipeline("cách biểu diễn số nguyên trong máy tính")

## 7. Test — So sánh có/không Query Rewrite

In [ ]:
q = "hệ điều hành có chức năng gì"

print("=" * 60)
print("--- VỚI Rewrite ---")
r_with = rag_pipeline(q, use_rewrite=True)

print("\n" + "=" * 60)
print("--- KHÔNG Rewrite ---")
r_without = rag_pipeline(q, use_rewrite=False)

## 8. Pipeline Stats

In [11]:
print("📊 PIPELINE SUMMARY")
print("=" * 40)
print(f"  Corpus: {searcher.corpus_size} chunks")
print(f"  Embedding dim: {searcher.embeddings.shape[1]}")
print(f"  BM25 vocab: {len(searcher.df)} terms")
print(f"  BM25 params: k1={searcher.k1}, b={searcher.b}")
print(f"  RRF k: {searcher.rrf_k}")
print(f"  Reranker: AITeamVN/Vietnamese_Reranker")
print(f"  Query Rewriter: {'ON' if USE_REWRITER else 'OFF'}")

📊 PIPELINE SUMMARY
  Corpus: 2348 chunks
  Embedding dim: 768
  BM25 vocab: 17139 terms
  BM25 params: k1=1.2, b=0.75
  RRF k: 60
  Reranker: AITeamVN/Vietnamese_Reranker
  Query Rewriter: ON
